# 07 — Explainability across all model families

For each language we pick **3–5 representative test examples** (mix of correct/incorrect predictions) and produce:

| Family       | Method                                                   |
|--------------|----------------------------------------------------------|
| TF-IDF + LR  | top coefficients (global) + LIME (local)                 |
| XGBoost      | SHAP TreeExplainer summary + force plot                  |
| BiLSTM       | LIME (model-agnostic)                                    |
| Transformer  | BertViz attention head view + Captum integrated gradients |
| LLM          | LLM's own self-explanation, side-by-side with SHAP       |

Outputs land in `results/explainability/`.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np, pandas as pd, matplotlib.pyplot as plt

from src import config as C, explain
from src.data_utils import get_split
from src.models import baseline, xgboost_model, transformer, llm

LANG = 'en'  # change as needed
_, _, test_df = get_split(LANG)

### Pick interesting examples

In [ ]:
lr_pipe = baseline.load(C.RESULTS_DIR / 'checkpoints' / f'baseline_{LANG}.joblib')
probs = lr_pipe.predict_proba(test_df['text'].tolist())[:, 1]
preds = (probs > 0.5).astype(int)

test_df = test_df.assign(prob_fake=probs, lr_pred=preds, correct=lambda d: (d['lr_pred'] == d['label']).astype(int))
samples = pd.concat([
    test_df[(test_df['correct'] == 1) & (test_df['label'] == 1)].head(2),
    test_df[(test_df['correct'] == 1) & (test_df['label'] == 0)].head(1),
    test_df[(test_df['correct'] == 0)].head(2),
]).reset_index(drop=True)
samples[['label', 'lr_pred', 'prob_fake', 'text']].head()

### LR — global top features (already exported in nb 02)

In [ ]:
fake_top, real_top = explain.explain_lr_global(lr_pipe, top_k=15)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].barh(fake_top['feature'][::-1], fake_top['coef'][::-1], color='#c44e52'); axes[0].set_title('LR — top FAKE features')
axes[1].barh(real_top['feature'][::-1], real_top['coef'][::-1], color='#4c72b0'); axes[1].set_title('LR — top REAL features')
fig.tight_layout()
fig.savefig(C.EXPLAIN_DIR / f'lr_top_features_{LANG}.png', dpi=150)
fig

### LR — LIME on a single article

In [ ]:
for i, row in samples.iterrows():
    exp = explain.explain_lime(lr_pipe, row['text'], num_features=12, num_samples=500)
    out = C.EXPLAIN_DIR / f'lr_lime_{LANG}_{i}.html'
    out.write_text(exp.as_html(), encoding='utf-8')
    print(f'wrote {out}')

### XGBoost — SHAP summary

In [ ]:
xgb_pipe = xgboost_model.load(C.RESULTS_DIR / 'checkpoints' / f'xgboost_{LANG}.joblib')
shap_pack = explain.explain_xgb_shap(xgb_pipe, samples)
import shap
shap.summary_plot(shap_pack['shap_values'], shap_pack['X'], feature_names=shap_pack['feature_names'], max_display=20, show=False)
plt.savefig(C.EXPLAIN_DIR / f'xgb_shap_summary_{LANG}.png', dpi=150, bbox_inches='tight')
plt.close()

### Transformer — Captum integrated gradients

In [ ]:
ckpt = C.RESULTS_DIR / 'checkpoints' / f'xlmr-base_{LANG}'
if ckpt.exists():
    bundle = transformer.load(ckpt)
    for i, row in samples.iterrows():
        attribs = explain.explain_transformer_ig(bundle['model'], bundle['tokenizer'], row['text'], target_class=int(row['label']))
        tab = explain.attribution_table(attribs, top_k=15)
        tab.to_csv(C.EXPLAIN_DIR / f'xlmr_ig_{LANG}_{i}.csv', index=False)
        print(f'\nExample {i} (label={row["label"]}):'); display(tab)
else:
    print('No transformer checkpoint found yet — run notebook 05 first.')

### LLM — self-explanation

In [ ]:
for i, row in samples.iterrows():
    label = 'fake' if row['lr_pred'] == 1 else 'real'
    rationale = explain.explain_llm(row['text'], predicted_label=label)
    out = C.EXPLAIN_DIR / f'llm_explanation_{LANG}_{i}.txt'
    out.write_text(rationale, encoding='utf-8')
    print(f'\n--- Example {i} (LR said {label}; true={row["label"]}) ---\n{rationale}\n')